In [1]:
from keras.preprocessing.image import ImageDataGenerator
from keras.preprocessing import image
from tensorflow.python.keras.models import Sequential
from tensorflow.python.keras.layers import Conv2D, MaxPooling2D
from tensorflow.python.keras.layers import Activation, Dropout, Flatten, Dense
from pathlib import Path

ModuleNotFoundError: No module named 'keras'

In [ ]:
BASE_DIR = Path(__file__).resolve().parent

# Создаем сверточную нейронную сеть

In [58]:
# Слой свертки, размер ядра 3х3, количество карт признаков - 32 шт., функция активации ReLU
# Слой подвыборки, выбор максимального значения из квадрата 2х2
model = Sequential()
model.add(Conv2D(32, (3, 3), input_shape=input_shape))
model.add(Activation('relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))

# Слой свертки, размер ядра 3х3, количество карт признаков - 32 шт., функция активации ReLU
# Слой подвыборки, выбор максимального значения из квадрата 2х2
model.add(Conv2D(32, (3, 3)))
model.add(Activation('relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))

# Слой свертки, размер ядра 3х3, количество карт признаков - 64 шт., функция активации ReLU.
# Слой подвыборки, выбор максимального значения из квадрата 2х2
model.add(Conv2D(64, (3, 3)))
model.add(Activation('relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))

# Слой преобразования из двумерного в одномерное представление
# Полносвязный слой, 64 нейрона, функция активации ReLU
# Слой Dropout
# Выходной слой, 1 нейрон, функция активации sigmoid
model.add(Flatten())
model.add(Dense(64))
model.add(Activation('relu'))
model.add(Dropout(0.5))
model.add(Dense(1))
model.add(Activation('sigmoid'))

# p.s. слои с 1 по 6 используются для выделения важных признаков в изображении, а слои с 7 по 10 - для классификации.

## STEP ONE: optimizer adam, epochs=10, train=500

3. Компилируем нейронную сеть. Используя функцию ошибки "binary_crossentropy", оптимизатор "adam", метрику качества "accuracy"

In [ ]:
model.compile(loss='binary_crossentropy', # функция ошибки бинарная кроссэнтропия (т.к. 2 класса)
              optimizer='adam',
              metrics=['accuracy'])

4. Задаем параметры для проведения первого эксперимента:

4.1. Указываем параметры изображений

4.2. Обучение на датасете размером 700 фото (500 тренировочные (250/250 каждого класса), 100 для валидации (50/50 каждого класса), 100 для тестирования (50/50 каждого класса)

4.3. Задаем количество эпох = 10

4.4. Задаем размер бача = 10

4.5. Создаем генератор изображений (генератор делит значения всех пикселей изображения на 255)

In [85]:
img_width, img_height = 150, 150 # размеры изображений
input_shape = (img_width, img_height, 3) # backend Tensorflow, формат хранения изображений channels_last (размерность тензора 150 на 150, 3 канала цвета)

train_dir = 'train' # каталог с данными для обучения
val_dir = 'val' # каталог с данными для проверки
test_dir = 'test' # каталог с данными для теста

epochs = 10 # количество эпох обучения
batch_size = 10 # размер бача

nb_train_samples = 500 # Кол-во изображений для обучения
nb_validation_samples = 100 # Кол-во изображений для проверки
nb_test_samples = 100 # Кол-во изображений для теста

datagen = ImageDataGenerator(rescale=1. / 255)

In [ ]:
train_dir = BASE_DIR / "data" / "Nuts700" / "train"
val_dir = BASE_DIR / "data" / "Nuts700" / "val"
test_dir = BASE_DIR / "data" / "Nuts700" / "test"

5.1. Работаем с каталогом train, val, test

In [87]:
train_generator = datagen.flow_from_directory(
    train_dir,
    target_size=(img_width, img_height),
    batch_size=batch_size,
    class_mode='binary')

Found 500 images belonging to 2 classes.


In [88]:
val_generator = datagen.flow_from_directory(
    val_dir,
    target_size=(img_width, img_height),
    batch_size=batch_size,
    class_mode='binary')

Found 100 images belonging to 2 classes.


In [89]:
test_generator = datagen.flow_from_directory(
    test_dir,
    target_size=(img_width, img_height),
    batch_size=batch_size,
    class_mode='binary')

Found 100 images belonging to 2 classes.


6. Обучаем нашу сверточную нейронную сеть, используя наши генераторы (train_generator700, val_generator700, test_generator700) и ранее введенные параметры обучения (optimizer (adam), metrics (accuracy), epochs, batch_size)

In [90]:
model.fit_generator(
    train_generator,
    steps_per_epoch=nb_train_samples // batch_size,
    epochs=epochs,
    validation_data=val_generator,
    validation_steps=nb_validation_samples // batch_size)

Epoch 1/10
50/50 [==============================] - 25s 507ms/step - loss: 0.1737 - accuracy: 0.9360 - val_loss: 3.6682 - val_accuracy: 0.5000
Epoch 2/10
50/50 [==============================] - 25s 502ms/step - loss: 0.0574 - accuracy: 0.9840 - val_loss: 1.1917 - val_accuracy: 0.5400
Epoch 3/10
50/50 [==============================] - 25s 509ms/step - loss: 0.0130 - accuracy: 0.9980 - val_loss: 2.4808 - val_accuracy: 0.5300
Epoch 4/10
50/50 [==============================] - 25s 505ms/step - loss: 0.0207 - accuracy: 0.9940 - val_loss: 1.5030 - val_accuracy: 0.5300
Epoch 5/10
50/50 [==============================] - 25s 499ms/step - loss: 0.0107 - accuracy: 0.9980 - val_loss: 2.6890 - val_accuracy: 0.4200
Epoch 6/10
50/50 [==============================] - 25s 503ms/step - loss: 0.0070 - accuracy: 0.9980 - val_loss: 2.8765 - val_accuracy: 0.4600
Epoch 7/10
50/50 [==============================] - 25s 503ms/step - loss: 0.0028 - accuracy: 0.9980 - val_loss: 3.0467 - val_accuracy: 0.5000

7. Осуществляем проверку на тестовых данных (метрика accuracy)

In [92]:
scores = model.evaluate_generator(test_generator, nb_test_samples // batch_size)
print("accuracy на тестовых данных: %.2f%%" % (scores[1]*100))

accuracy на тестовых данных: 50.00%


Таким образом, исходя из метрики accuracy на тестовых данных: 50.00% мы делаем вывод что наша сверточная нейронная сеть используя 500 фото для обучения (по 250 каждого класса true/false) и количество эпох обучения = 10 **НЕДООБУЧИЛАСЬ** и показала сравнительно плохой результат предсказания

## STEP TWO: optimizer adam, epochs=10, train=4000 (остальные метрики НЕ изменяем)

In [95]:
model.compile(loss='binary_crossentropy', # функция ошибки бинарная кроссэнтропия (т.к. 2 класса)
              optimizer='adam', # опитизатор
              metrics=['accuracy']) # метрика

In [100]:
img_width, img_height = 150, 150 # размеры изображений
input_shape = (img_width, img_height, 3) # backend Tensorflow, формат хранения изображений channels_last (размерность тензора 150 на 150, 3 канала цвета)

train_dir2 = 'train' # каталог с данными для обучения
val_dir2 = 'val' # каталог с данными для проверки
test_dir2 = 'test' # каталог с данными для теста

epochs = 10 # количество эпох обучения
batch_size = 10 # размер бача

nb_train_samples2 = 4000 # Кол-во изображений для обучения
nb_validation_samples2 = 500 # Кол-во изображений для проверки
nb_test_samples2 = 500 # Кол-во изображений для теста

datagen2 = ImageDataGenerator(rescale=1. / 255)

In [ ]:
train_dir2 = BASE_DIR / "data" / "Nuts5000" / "train"
val_dir2 = BASE_DIR / "data" / "Nuts5000" / "val"
test_dir2 = BASE_DIR / "data" / "Nuts5000" / "test"

In [102]:
train_generator2 = datagen2.flow_from_directory(
    train_dir2,
    target_size=(img_width, img_height),
    batch_size=batch_size,
    class_mode='binary')

Found 4000 images belonging to 2 classes.


In [103]:
val_generator2 = datagen2.flow_from_directory(
    val_dir2,
    target_size=(img_width, img_height),
    batch_size=batch_size,
    class_mode='binary')

Found 500 images belonging to 2 classes.


In [104]:
test_generator2 = datagen2.flow_from_directory(
    test_dir2,
    target_size=(img_width, img_height),
    batch_size=batch_size,
    class_mode='binary')

Found 500 images belonging to 2 classes.


In [105]:
model.fit_generator(
    train_generator2,
    steps_per_epoch=nb_train_samples2 // batch_size,
    epochs=epochs,
    validation_data=val_generator2,
    validation_steps=nb_validation_samples2 // batch_size)

Epoch 1/10
400/400 [==============================] - 187s 467ms/step - loss: 0.2113 - accuracy: 0.9258 - val_loss: 0.3744 - val_accuracy: 0.8800
Epoch 2/10
400/400 [==============================] - 185s 464ms/step - loss: 0.1188 - accuracy: 0.9603 - val_loss: 0.3815 - val_accuracy: 0.8580
Epoch 3/10
400/400 [==============================] - 184s 460ms/step - loss: 0.0656 - accuracy: 0.9808 - val_loss: 0.5761 - val_accuracy: 0.8440
Epoch 4/10
400/400 [==============================] - 184s 462ms/step - loss: 0.0529 - accuracy: 0.9837 - val_loss: 0.6154 - val_accuracy: 0.8120
Epoch 5/10
400/400 [==============================] - 185s 463ms/step - loss: 0.0470 - accuracy: 0.9847 - val_loss: 0.8097 - val_accuracy: 0.7700
Epoch 6/10
400/400 [==============================] - 185s 463ms/step - loss: 0.1000 - accuracy: 0.9663 - val_loss: 0.6629 - val_accuracy: 0.8280
Epoch 7/10
400/400 [==============================] - 184s 462ms/step - loss: 0.0727 - accuracy: 0.9793 - val_loss: 0.7283 -

In [106]:
scores = model.evaluate_generator(test_generator2, nb_test_samples2 // batch_size)
print("accuracy на тестовых данных: %.2f%%" % (scores[1]*100))

accuracy на тестовых данных: 94.80%


Таким образом, исходя из метрики accuracy на тестовых данных: 94.80% мы делаем вывод что наша сверточная нейронная сеть используя 4000 фото для обучения (по 2000 каждого класса true/false) и количество эпох обучения = 10 **ОБУЧИЛАСЬ ДОСТАТОЧНО** и показала сравнительно высокий результат предсказания

## STEP THREE: optimizer adam, epochs=20, train=4000 (остальные метрики НЕ изменяем)

Исходя из STEP TWO мы поняли, что увеличение размере датасета при сохранении количества эпох обучения существенно улучшает качество нашей модели. Проверяем гипотезу: увеличение количества эпох покажет существеннуый рост accuracy

In [109]:
model.compile(loss='binary_crossentropy', # функция ошибки бинарная кроссэнтропия (т.к. 2 класса)
              optimizer='adam', # опитизатор
              metrics=['accuracy']) # метрика

In [ ]:
img_width, img_height = 150, 150 # размеры изображений
input_shape = (img_width, img_height, 3) # backend Tensorflow, формат хранения изображений channels_last (размерность тензора 150 на 150, 3 канала цвета)

train_dir3 = 'train' # каталог с данными для обучения
val_dir3 = 'val' # каталог с данными для проверки
test_dir3 = 'test' # каталог с данными для теста

epochs = 20 # количество эпох обучения
batch_size = 10 # размер бача

nb_train_samples3 = 4000 # Кол-во изображений для обучения
nb_validation_samples3 = 500 # Кол-во изображений для проверки
nb_test_samples3 = 500 # Кол-во изображений для теста

datagen3 = ImageDataGenerator(rescale=1. / 255)

In [ ]:
train_dir3 = BASE_DIR / "data" / "Nuts5000" / "train"
val_dir3 = BASE_DIR / "data" / "Nuts5000" / "val"
test_dir3 = BASE_DIR / "data" / "Nuts5000" / "test"

NameError: name 'Path' is not defined

In [115]:
train_generator3 = datagen3.flow_from_directory(
    train_dir3,
    target_size=(img_width, img_height),
    batch_size=batch_size,
    class_mode='binary')

Found 4000 images belonging to 2 classes.


In [116]:
val_generator3 = datagen3.flow_from_directory(
    val_dir3,
    target_size=(img_width, img_height),
    batch_size=batch_size,
    class_mode='binary')

Found 500 images belonging to 2 classes.


In [117]:
test_generator3 = datagen3.flow_from_directory(
    test_dir3,
    target_size=(img_width, img_height),
    batch_size=batch_size,
    class_mode='binary')

Found 500 images belonging to 2 classes.


In [118]:
model.fit_generator(
    train_generator3,
    steps_per_epoch=nb_train_samples3 // batch_size,
    epochs=epochs,
    validation_data=val_generator3,
    validation_steps=nb_validation_samples3 // batch_size)

Epoch 1/20
400/400 [==============================] - 187s 467ms/step - loss: 0.0519 - accuracy: 0.9885 - val_loss: 1.0107 - val_accuracy: 0.7860
Epoch 2/20
400/400 [==============================] - 186s 466ms/step - loss: 0.0384 - accuracy: 0.9895 - val_loss: 0.8393 - val_accuracy: 0.8560
Epoch 3/20
400/400 [==============================] - 186s 466ms/step - loss: 0.0162 - accuracy: 0.9965 - val_loss: 0.8808 - val_accuracy: 0.8300
Epoch 4/20
400/400 [==============================] - 186s 467ms/step - loss: 0.0225 - accuracy: 0.9933 - val_loss: 1.0127 - val_accuracy: 0.7880
Epoch 5/20
400/400 [==============================] - 186s 466ms/step - loss: 0.0118 - accuracy: 0.9965 - val_loss: 1.1557 - val_accuracy: 0.7480
Epoch 6/20
400/400 [==============================] - 186s 466ms/step - loss: 0.1018 - accuracy: 0.9818 - val_loss: 0.4771 - val_accuracy: 0.8480
Epoch 7/20
400/400 [==============================] - 186s 466ms/step - loss: 0.0175 - accuracy: 0.9935 - val_loss: 1.2550 -

In [119]:
scores = model.evaluate_generator(test_generator3, nb_test_samples3 // batch_size)
print("accuracy на тестовых данных: %.2f%%" % (scores[1]*100))

accuracy на тестовых данных: 91.80%


Таким образом, исходя из метрики accuracy на тестовых данных: 91.80% мы делаем вывод что наша сверточная нейронная сеть используя 4000 фото для обучения (по 2000 каждого класса true/false) и количество эпох обучения = 20 а не 10 как в 'STEP TWO' **ПЕРЕОБУЧИЛАСЬ** и имеет сравнительно худший результат предсказания. Исходя из теста проведенного в 'STEP THREE' полагаем, что дальнейшее увеличение количества эпох обучения > 20 не приведет к улучшению качества модели.

## STEP FOUR: optimizer SGD, epochs=10, train=500 (остальные метрики НЕ изменяем)

Проведем сравнение качества обучения нашей сверточной сети используя другой метод оптимизации целевой функции = стохастический градиентный спуск (SGD). Полученные в ходе результаты accuracy сравним с: STEP ONE (optimizer adam, epochs=10, train=500). Таким образом, сделаем вывод какой лучше использовать метод оптимизации.

In [127]:
model.compile(loss='binary_crossentropy', # функция ошибки бинарная кроссэнтропия (т.к. 2 класса)
              optimizer='SGD', # опитизатор
              metrics=['accuracy']) # метрика

In [128]:
img_width, img_height = 150, 150 # размеры изображений
input_shape = (img_width, img_height, 3) # backend Tensorflow, формат хранения изображений channels_last (размерность тензора 150 на 150, 3 канала цвета)

train_dir4 = 'train' # каталог с данными для обучения
val_dir4 = 'val' # каталог с данными для проверки
test_dir4 = 'test' # каталог с данными для теста

epochs = 10 # количество эпох обучения
batch_size = 10 # размер бача

nb_train_samples4 = 500 # Кол-во изображений для обучения
nb_validation_samples4 = 100 # Кол-во изображений для проверки
nb_test_samples4 = 100 # Кол-во изображений для теста

datagen4 = ImageDataGenerator(rescale=1. / 255)

In [ ]:
train_dir4 = BASE_DIR / "data" / "Nuts700" / "train"
val_dir4 = BASE_DIR / "data" / "Nuts700" / "val"
test_dir4 = BASE_DIR / "data" / "Nuts700" / "test"

In [130]:
train_generator4 = datagen4.flow_from_directory(
    train_dir4,
    target_size=(img_width, img_height),
    batch_size=batch_size,
    class_mode='binary')

Found 500 images belonging to 2 classes.


In [131]:
val_generator4 = datagen4.flow_from_directory(
    val_dir4,
    target_size=(img_width, img_height),
    batch_size=batch_size,
    class_mode='binary')

Found 100 images belonging to 2 classes.


In [132]:
test_generator4 = datagen4.flow_from_directory(
    test_dir4,
    target_size=(img_width, img_height),
    batch_size=batch_size,
    class_mode='binary')

Found 100 images belonging to 2 classes.


In [133]:
model.fit_generator(
    train_generator4,
    steps_per_epoch=nb_train_samples4 // batch_size,
    epochs=epochs,
    validation_data=val_generator4,
    validation_steps=nb_validation_samples4 // batch_size)

Epoch 1/10
50/50 [==============================] - 26s 516ms/step - loss: 978.8244 - accuracy: 0.4800 - val_loss: 0.7167 - val_accuracy: 0.5000
Epoch 2/10
50/50 [==============================] - 25s 505ms/step - loss: 0.7142 - accuracy: 0.5000 - val_loss: 0.7078 - val_accuracy: 0.5100
Epoch 3/10
50/50 [==============================] - 24s 492ms/step - loss: 0.7097 - accuracy: 0.5000 - val_loss: 0.7280 - val_accuracy: 0.4400
Epoch 4/10
50/50 [==============================] - 24s 493ms/step - loss: 0.7061 - accuracy: 0.5000 - val_loss: 0.6984 - val_accuracy: 0.5200
Epoch 5/10
50/50 [==============================] - 24s 494ms/step - loss: 0.7032 - accuracy: 0.5000 - val_loss: 0.7125 - val_accuracy: 0.4600
Epoch 6/10
50/50 [==============================] - 24s 490ms/step - loss: 0.7011 - accuracy: 0.5000 - val_loss: 0.6976 - val_accuracy: 0.5100
Epoch 7/10
50/50 [==============================] - 24s 494ms/step - loss: 0.6993 - accuracy: 0.5000 - val_loss: 0.6964 - val_accuracy: 0.51

In [134]:
scores = model.evaluate_generator(test_generator4, nb_test_samples4 // batch_size)
print("accuracy на тестовых данных: %.2f%%" % (scores[1]*100))

accuracy на тестовых данных: 50.00%


Таким образом, исходя из метрики accuracy на тестовых данных: 50.00% мы делаем вывод что наша сверточная нейронная сеть используя 500 фото для обучения (по 250 каждого класса true/false) и количество эпох обучения = 10 НЕДООБУЧИЛАСЬ и показала схожий результат с STEP ONE (optimizer adam, epochs=10, train=500), где в качестве метода оптимизации мы использовали 'Adam'

## STEP FIVE: optimizer RMSProp, epochs=10, train=500 (остальные метрики НЕ изменяем)

p.s. Проведем сравнение качества обучения нашей сверточной сети используя другой метод оптимизации целевой функции = root mean square propagation (RMSProp). Полученные в ходе результаты accuracy сравним с: STEP ONE (optimizer adam, epochs=10, train=500). Таким образом, сделаем вывод какой лучше использовать метод оптимизации.

In [135]:
model.compile(loss='binary_crossentropy', # функция ошибки бинарная кроссэнтропия (т.к. 2 класса)
              optimizer='RMSProp', # опитизатор
              metrics=['accuracy']) # метрика

In [136]:
img_width, img_height = 150, 150 # размеры изображений
input_shape = (img_width, img_height, 3) # backend Tensorflow, формат хранения изображений channels_last (размерность тензора 150 на 150, 3 канала цвета)

train_dir5 = 'train' # каталог с данными для обучения
val_dir5 = 'val' # каталог с данными для проверки
test_dir5 = 'test' # каталог с данными для теста

epochs = 10 # количество эпох обучения
batch_size = 10 # размер бача

nb_train_samples5 = 500 # Кол-во изображений для обучения
nb_validation_samples5 = 100 # Кол-во изображений для проверки
nb_test_samples5 = 100 # Кол-во изображений для теста

datagen5 = ImageDataGenerator(rescale=1. / 255)

In [ ]:
train_dir5 = BASE_DIR / "data" / "Nuts700" / "train"
val_dir5 = BASE_DIR / "data" / "Nuts700" / "val"
test_dir5 = BASE_DIR / "data" / "Nuts700" / "test"

In [138]:
train_generator5 = datagen5.flow_from_directory(
    train_dir5,
    target_size=(img_width, img_height),
    batch_size=batch_size,
    class_mode='binary')

Found 500 images belonging to 2 classes.


In [139]:
val_generator5 = datagen5.flow_from_directory(
    val_dir5,
    target_size=(img_width, img_height),
    batch_size=batch_size,
    class_mode='binary')

Found 100 images belonging to 2 classes.


In [140]:
test_generator5 = datagen5.flow_from_directory(
    test_dir5,
    target_size=(img_width, img_height),
    batch_size=batch_size,
    class_mode='binary')

Found 100 images belonging to 2 classes.


In [141]:
model.fit_generator(
    train_generator5,
    steps_per_epoch=nb_train_samples5 // batch_size,
    epochs=epochs,
    validation_data=val_generator5,
    validation_steps=nb_validation_samples5 // batch_size)

Epoch 1/10
50/50 [==============================] - 25s 499ms/step - loss: 0.6955 - accuracy: 0.5000 - val_loss: 0.6952 - val_accuracy: 0.5000
Epoch 2/10
50/50 [==============================] - 24s 495ms/step - loss: 0.6952 - accuracy: 0.5000 - val_loss: 0.6962 - val_accuracy: 0.4900
Epoch 3/10
50/50 [==============================] - 24s 495ms/step - loss: 0.6950 - accuracy: 0.5000 - val_loss: 0.6936 - val_accuracy: 0.5100
Epoch 4/10
50/50 [==============================] - 24s 494ms/step - loss: 0.6947 - accuracy: 0.5000 - val_loss: 0.6945 - val_accuracy: 0.5000
Epoch 5/10
50/50 [==============================] - 24s 493ms/step - loss: 0.6945 - accuracy: 0.5000 - val_loss: 0.6953 - val_accuracy: 0.4900
Epoch 6/10
50/50 [==============================] - 24s 494ms/step - loss: 0.6943 - accuracy: 0.5000 - val_loss: 0.6905 - val_accuracy: 0.5400
Epoch 7/10
50/50 [==============================] - 24s 493ms/step - loss: 0.6942 - accuracy: 0.5000 - val_loss: 0.6932 - val_accuracy: 0.5100

In [142]:
scores = model.evaluate_generator(test_generator5, nb_test_samples5 // batch_size)
print("accuracy на тестовых данных: %.2f%%" % (scores[1]*100))

accuracy на тестовых данных: 50.00%


Таким образом, исходя из метрики accuracy на тестовых данных: 50.00% мы делаем вывод что наша сверточная нейронная сеть используя 500 фото для обучения (по 250 каждого класса true/false) и количество эпох обучения = 10 НЕДООБУЧИЛАСЬ и показала схожий результат с STEP ONE (optimizer adam, epochs=10, train=500), где в качестве метода оптимизации мы использовали 'Adam'

# Вывод: в ходе тестов, мы пришли к выводу, что наиболее оптимальные результаты обучения нашей сверточной сети были получены используя параметры в  STEP TWO (optimizer adam, epochs=10, train=4000). Методы оптимизации RMSProp и SGD не дали прирост accuracy.